In [1]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np

# 👇 Use the same PROJECT_ROOT path you used in Step 2
PROJECT_ROOT = Path(r"C:\Users\kjdac\OneDrive\Desktop\MerchantRisk-project")
DB_PATH = PROJECT_ROOT / "db" / "merchant_risk.db"

con = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("SELECT * FROM merchant_features;", con)
con.close()

print("Rows:", len(df), "Cols:", df.shape[1])
display(df.head())


Rows: 500 Cols: 27


,merchant_id,merchant_name,industry,country,onboard_date,onboarding_channel,txn_total,txn_approved,txn_declined,approval_rate,...,dispute_count,dispute_amount,dispute_lost_count,dispute_rate,distinct_devices,distinct_ips,txns_per_device,txns_per_ip,active_days,avg_approved_txns_per_day
0,1,Merchant_0001,travel,US,2025-06-18,sales_assisted,497,397,100,0.7988,...,3,1022.28,1,0.0076,397,397,1.000,1.0,158,2.51
1,2,Merchant_0002,travel,US,2025-12-25,sales_assisted,209,182,27,0.8708,...,0,0.00,0,0.0000,182,182,1.000,1.0,105,1.73
2,3,Merchant_0003,marketplace,US,2025-07-30,self_serve,488,392,96,0.8033,...,3,287.96,1,0.0077,392,392,1.000,1.0,161,2.43
3,4,Merchant_0004,fashion,GB,2025-09-19,self_serve,222,202,20,0.9099,...,3,302.54,1,0.0149,202,202,1.000,1.0,122,1.66
4,5,Merchant_0005,electronics,US,2025-10-25,self_serve,447,342,105,0.7651,...,6,1927.61,0,0.0175,341,342,1.003,1.0,156,2.19


In [2]:
# Some rates can be NULL if a merchant has 0 approved txns (rare but possible)
rate_cols = ["approval_rate", "refund_rate", "dispute_rate", "avg_approved_txns_per_day", "txns_per_device", "txns_per_ip"]
df[rate_cols] = df[rate_cols].fillna(0)

print(df[rate_cols].describe())


       approval_rate  refund_rate  dispute_rate  avg_approved_txns_per_day  \
count     500.000000   500.000000    500.000000                 500.000000   
mean        0.842716     0.024769      0.011244                   2.120740   
std         0.057683     0.013923      0.008241                   0.461575   
min         0.662000     0.000000      0.000000                   1.260000   
25%         0.803275     0.014575      0.005500                   1.717500   
50%         0.848550     0.022100      0.009600                   2.120000   
75%         0.886100     0.032425      0.015850                   2.462500   
max         0.962300     0.076600      0.053000                   3.210000   

       txns_per_device  txns_per_ip  
count        500.00000        500.0  
mean           1.00021          1.0  
std            0.00086          0.0  
min            1.00000          1.0  
25%            1.00000          1.0  
50%            1.00000          1.0  
75%            1.00000         

In [3]:
def points_for_approval_rate(x):
    # lower approval rate => more risk points
    if x < 0.70: return 35
    if x < 0.80: return 20
    if x < 0.90: return 10
    return 0

def points_for_refund_rate(x):
    if x > 0.06: return 25
    if x > 0.03: return 12
    if x > 0.015: return 6
    return 0

def points_for_dispute_rate(x):
    if x > 0.03: return 35
    if x > 0.015: return 18
    if x > 0.007: return 8
    return 0

def points_for_velocity(x):
    # avg approved txns per day
    if x > 20: return 15
    if x > 12: return 8
    return 0

def points_for_device_reuse(x):
    # txns per device high => possible reuse/automation
    if x > 18: return 15
    if x > 12: return 8
    return 0

def points_for_ip_reuse(x):
    if x > 20: return 10
    if x > 14: return 5
    return 0

df["pts_approval"] = df["approval_rate"].apply(points_for_approval_rate)
df["pts_refund"] = df["refund_rate"].apply(points_for_refund_rate)
df["pts_dispute"] = df["dispute_rate"].apply(points_for_dispute_rate)
df["pts_velocity"] = df["avg_approved_txns_per_day"].apply(points_for_velocity)
df["pts_device"] = df["txns_per_device"].apply(points_for_device_reuse)
df["pts_ip"] = df["txns_per_ip"].apply(points_for_ip_reuse)

df["risk_score"] = (
    df["pts_approval"] +
    df["pts_refund"] +
    df["pts_dispute"] +
    df["pts_velocity"] +
    df["pts_device"] +
    df["pts_ip"]
)

display(df[[
    "merchant_id","merchant_name","industry","country",
    "approval_rate","refund_rate","dispute_rate",
    "avg_approved_txns_per_day","txns_per_device","txns_per_ip",
    "risk_score"
]].head())


,merchant_id,merchant_name,industry,country,approval_rate,refund_rate,dispute_rate,avg_approved_txns_per_day,txns_per_device,txns_per_ip,risk_score
0,1,Merchant_0001,travel,US,0.7988,0.0302,0.0076,2.51,1.000,1.0,40
1,2,Merchant_0002,travel,US,0.8708,0.0275,0.0000,1.73,1.000,1.0,16
2,3,Merchant_0003,marketplace,US,0.8033,0.0281,0.0077,2.43,1.000,1.0,24
3,4,Merchant_0004,fashion,GB,0.9099,0.0050,0.0149,1.66,1.000,1.0,8
4,5,Merchant_0005,electronics,US,0.7651,0.0526,0.0175,2.19,1.003,1.0,50


In [4]:
def decision_from_score(s):
    if s >= 70:
        return "decline"
    if s >= 35:
        return "manual_review"
    return "approve"

df["decision"] = df["risk_score"].apply(decision_from_score)

print(df["decision"].value_counts(normalize=True).round(3))
display(df.sort_values("risk_score", ascending=False).head(10)[
    ["merchant_id","merchant_name","risk_score","decision","approval_rate","refund_rate","dispute_rate"]
])


decision
approve          0.762
manual_review    0.228
decline          0.010
Name: proportion, dtype: float64


,merchant_id,merchant_name,risk_score,decision,approval_rate,refund_rate,dispute_rate
67,68,Merchant_0068,95,decline,0.6732,0.0640,0.0349
37,38,Merchant_0038,82,decline,0.6657,0.0563,0.0390
193,194,Merchant_0194,82,decline,0.6749,0.0307,0.0399
80,81,Merchant_0081,80,decline,0.7156,0.0621,0.0435
167,168,Merchant_0168,78,decline,0.6620,0.0766,0.0213
455,456,Merchant_0456,67,manual_review,0.7421,0.0390,0.0319
326,327,Merchant_0327,67,manual_review,0.7204,0.0476,0.0310
470,471,Merchant_0471,67,manual_review,0.7861,0.0380,0.0316
10,11,Merchant_0011,67,manual_review,0.7406,0.0452,0.0339
172,173,Merchant_0173,67,manual_review,0.7733,0.0376,0.0301


In [5]:
reason_cols = ["pts_approval","pts_refund","pts_dispute","pts_velocity","pts_device","pts_ip"]

def top_reasons(row, k=2):
    pairs = sorted([(c, row[c]) for c in reason_cols], key=lambda x: x[1], reverse=True)
    top = [p for p in pairs if p[1] > 0][:k]
    if not top:
        return "low_risk_signals"
    return ", ".join([name.replace("pts_", "") for name, _ in top])

df["top_reasons"] = df.apply(top_reasons, axis=1)

display(df[["merchant_id","merchant_name","risk_score","decision","top_reasons"]].head(10))


,merchant_id,merchant_name,risk_score,decision,top_reasons
0,1,Merchant_0001,40,manual_review,"approval, refund"
1,2,Merchant_0002,16,approve,"approval, refund"
2,3,Merchant_0003,24,approve,"approval, dispute"
3,4,Merchant_0004,8,approve,dispute
4,5,Merchant_0005,50,manual_review,"approval, dispute"
5,6,Merchant_0006,6,approve,refund
6,7,Merchant_0007,50,manual_review,"approval, dispute"
7,8,Merchant_0008,10,approve,approval
8,9,Merchant_0009,10,approve,approval
9,10,Merchant_0010,34,approve,"dispute, approval"


In [6]:
out_path = PROJECT_ROOT / "outputs" / "onboarding_scorecard_output.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

cols_for_tableau = [
    "merchant_id","merchant_name","industry","country","onboard_date","onboarding_channel",
    "txn_total","txn_approved","txn_declined","approval_rate",
    "approved_txn_count","avg_amount","gmv","max_amount",
    "refund_count","refund_amount","refund_rate",
    "dispute_count","dispute_amount","dispute_lost_count","dispute_rate",
    "distinct_devices","distinct_ips","txns_per_device","txns_per_ip",
    "active_days","avg_approved_txns_per_day",
    "risk_score","decision","top_reasons"
]

df[cols_for_tableau].to_csv(out_path, index=False)
print("✅ Saved:", out_path)


✅ Saved: C:\Users\kjdac\OneDrive\Desktop\MerchantRisk-project\outputs\onboarding_scorecard_output.csv
